# 0. 분석 패키지 설치

In [1]:
import sys
print(sys.version)

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]


In [8]:
# conda install -c conda-forge jpype1

In [9]:
# !pip install konlpy

# 1.라이브러리 불러오기

In [2]:
import numpy as np  
import pandas as pd
import re
import string
import konlpy
from konlpy.tag import Okt

# 2.형태소

- Okt, 코모란, 한나눔, 꼬고마, 메캅 등 5개 오픈소스 형태소 분석기를 파이썬 환경에서 사용할 수 있도록 인터페이스를 통일한 한국어 자연처리 패키지

    - Otk
    - Komoran
    - Hannanum
    - Kkma
    - Mecab

In [3]:
import os
# os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.11"

from konlpy.tag import Kkma

In [4]:
tokenizer = Kkma()

In [5]:
# morphs() 함수 : 입력된 문장을 단어(형태소) 단위로 나누어 리스트로 반환
sentence = '아버지가방에들어가신다.'
tokenizer.morphs(sentence)

['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']

In [6]:
# pos() 함수 : 입력된 문장을 형태소 단위로 나누고 각 형태소의 품사 태그를 함께 반환
tokenizer.pos(sentence)

[('아버지', 'NNG'),
 ('가방', 'NNG'),
 ('에', 'JKM'),
 ('들어가', 'VV'),
 ('시', 'EPH'),
 ('ㄴ다', 'EFN'),
 ('.', 'SF')]

In [7]:
from konlpy.tag import Okt, Komoran, Hannanum, Kkma

In [8]:
# 형태소 분석기 함수 만들기
def get_tokenizer(tokenizer_name):
    if tokenizer_name == 'Okt':
        tokenizer = Okt()
    elif tokenizer_name == 'Komoran':
        tokenizer = Komoran()
    elif tokenizer_name == 'Hannanum':
        tokenizer = Hannanum()
    else:
        tokenizer = Kkma()
    return tokenizer

In [17]:
tokenizer = get_tokenizer('Okt')
print('Okt 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Okt 형태소 분석기
['아버지', '가방', '에', '들어가신다', '.']
[('아버지', 'Noun'), ('가방', 'Noun'), ('에', 'Josa'), ('들어가신다', 'Verb'), ('.', 'Punctuation')]


In [18]:
tokenizer = get_tokenizer('Komoran')
print('Komoran 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Komoran 형태소 분석기
['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']
[('아버지', 'NNG'), ('가방', 'NNP'), ('에', 'JKB'), ('들어가', 'VV'), ('시', 'EP'), ('ㄴ다', 'EF'), ('.', 'SF')]


In [19]:
tokenizer = get_tokenizer('Hannanum')
print('Hannanum 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Hannanum 형태소 분석기
['아버지가방에들어가', '이', '시ㄴ다', '.']
[('아버지가방에들어가', 'N'), ('이', 'J'), ('시ㄴ다', 'E'), ('.', 'S')]


In [20]:
tokenizer = get_tokenizer('Kkma')
print('Kkma 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Kkma 형태소 분석기
['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']
[('아버지', 'NNG'), ('가방', 'NNG'), ('에', 'JKM'), ('들어가', 'VV'), ('시', 'EPH'), ('ㄴ다', 'EFN'), ('.', 'SF')]


# 3.텍스트 전처리

## 3.1.데이터 수집 및 텍스트 정제
(특수 문자 제거, 소문자 변환)

In [23]:
df = pd.read_csv("result.csv", encoding="utf-8-sig")

reviews = (
    df.iloc[:, 3]
    .astype(str)
    .str.replace(r"[^ㄱ-ㅎㅏ-ㅣ가-힣\s]", "", regex=True)
    .tolist()
)

print(reviews[0])

오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음


In [25]:
# 한글과 공백만 남기기

re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣\s]", "", reviews[0])

'오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음'

In [26]:
def clean_text(text):
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text)
    return text

In [27]:
# reviews 리스트를 for문 돌려서 텍스트 클리닝 작업
cleaned_reviews = [clean_text(review) for review in reviews]
cleaned_reviews

['오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음',
 '좋아요',
 '편리함',
 '재미있어요',
 '잘모르겠음',
 '수수료없고 매일 이자주고 다 좋은데 방송봤는데 포인트 안들어오구ㅡㅡ업뎃하래서 했는데 계속 업뎃해야 이용가능하다 뜨고 결국 업뎃만 벌써 번하는거 같은데 언제 반영되나요',
 '매우좋음',
 '통장 잔액을 보려고 할때 재부팅이 느린건지 잔액이 얼마 남았는지 확인하는데 많은 시간이 걸림',
 '재미삼아 날마다 앱 열고 하고있습니다',
 '포인트를 원 원 주지 말고 원이라도 주시오ㅠㅠ',
 '좋습니다',
 '공유합니다',
 '사용하기 편리해서 좋아요  ',
 '아주좋아요',
 '앱 실행하면 상단에 대출광고 너무 많이 뜹니다 표시 눌러도 계속 뜹니다 길거리에서 명함 뿌리는 사채업자랑 뭐가 다릅니까',
 '좋아요',
 '일주일 방문 미션할때 가끔은 정답을 맞혀도 정답이 인정 안되고 그럴때도 있는데 즐겁게 토스앱 이용 하고 있습니다',
 '이번주미션이안열려요',
 '토스가작동하지않내요빨리고쳐주세요',
 '편리해요',
 '업데이트하라고 무한반복 열기를 눌렀는데도 계속 제자리 서버를 안정화 시키고 실행시키시지요',
 '환급해준다면서 준비서류 안내도없이 순서대로 결제하니까 마지막에 서류요구하네 이새키들이 서류있었으면 종소세 경정청구했겠지 장난까냐 부동산없어져서 서류구하지도못하는데 염장지르네 돈많이벌어먹으세요 엔화로 장난치더니 이제 환급으로 장난질이네',
 '만보기 복권 다 만원 만원 기본이 원인데 저 원 원 원 많이주면 윈입니다 매일 키로 걸으면뭐합니까 너무합니다 사람 차별하나요 아님 저 무시하는겁니까',
 '유용하게 사용중입니다',
 '토스 측에선 돈독 올랐는지 어떻게든 환급금 수수료 때먹구 싶어서 알림 설정 전부 해제해도 거슬리게 배너 띄우고 욕나오게 만듭니다',
 '공유해요',
 '미션 클릭하면 업데이트 하라고만 뜨고 업데이트가 안됨',
 '조아요',
 '월 일에 이번 주 미션이 오늘의 포인트 미션으로 바뀌었는데 언제부터 이

## 3.2.토큰화 (Tokenization)

In [28]:
# Okt 형태소 분석기 초기화
okt = Okt()

In [29]:
# Okt 형태소 분석기를 사용하여 토큰화
okt_tokenized_reviews = [okt.morphs(review) for review in cleaned_reviews]
print('Okt 형태소 단위 토큰화')
print(okt_tokenized_reviews)

Okt 형태소 단위 토큰화
[['오늘', '의', '포인트', '를', '하려면', '업데이트', '를', '하라', '고', '문구', '가', '뜨는데', '업데이트', '가', '되지', '않음'], ['좋아요'], ['편리함'], ['재미있어요'], ['잘', '모르겠음'], ['수수료', '없고', '매일', '이', '자주', '고', '다', '좋은데', '방송', '봤는데', '포인트', '안', '들어오구', 'ㅡㅡ', '업뎃', '하래', '서', '했는데', '계속', '업뎃해', '야', '이용', '가능하다', '뜨고', '결국', '업뎃', '만', '벌써', '번하는거', '같은데', '언제', '반영', '되나요'], ['매우', '좋음'], ['통장', '잔액', '을', '보려고', '할', '때', '재부팅', '이', '느린', '건지', '잔액', '이', '얼마', '남았는지', '확인', '하는데', '많은', '시간', '이', '걸림'], ['재미', '삼아', '날', '마다', '앱', '열고', '하고있습니다'], ['포인트', '를', '원', '원', '주지', '말고', '원', '이라도', '주', '시오', 'ㅠㅠ'], ['좋습니다'], ['공유', '합니다'], ['사용', '하기', '편리해서', '좋아요'], ['아주', '좋아요'], ['앱', '실행', '하면', '상단', '에', '대출', '광고', '너무', '많이', '뜹니다', '표시', '눌러도', '계속', '뜹니다', '길거리', '에서', '명함', '뿌리는', '사채', '업자', '랑', '뭐', '가', '다릅니까'], ['좋아요'], ['일주일', '방문', '미션', '할', '때', '가끔', '은', '정답', '을', '맞혀도', '정답', '이', '인정', '안되고', '그럴', '때', '도', '있는데', '즐겁게', '토스', '앱', '이용', '하고', '있습니다'], ['이번', '주', '미션',

## 3.3.불용어 제거

In [30]:
# 불용어 목록
stop_words = ['이', '정말', '완전', '조금', '는', '꼭']

In [31]:
reviews_no_stopwords = []

for tokens in okt_tokenized_reviews:
    filtered_tokens = []
    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)
    reviews_no_stopwords.append(filtered_tokens)

print(reviews_no_stopwords)


[['오늘', '의', '포인트', '를', '하려면', '업데이트', '를', '하라', '고', '문구', '가', '뜨는데', '업데이트', '가', '되지', '않음'], ['좋아요'], ['편리함'], ['재미있어요'], ['잘', '모르겠음'], ['수수료', '없고', '매일', '자주', '고', '다', '좋은데', '방송', '봤는데', '포인트', '안', '들어오구', 'ㅡㅡ', '업뎃', '하래', '서', '했는데', '계속', '업뎃해', '야', '이용', '가능하다', '뜨고', '결국', '업뎃', '만', '벌써', '번하는거', '같은데', '언제', '반영', '되나요'], ['매우', '좋음'], ['통장', '잔액', '을', '보려고', '할', '때', '재부팅', '느린', '건지', '잔액', '얼마', '남았는지', '확인', '하는데', '많은', '시간', '걸림'], ['재미', '삼아', '날', '마다', '앱', '열고', '하고있습니다'], ['포인트', '를', '원', '원', '주지', '말고', '원', '이라도', '주', '시오', 'ㅠㅠ'], ['좋습니다'], ['공유', '합니다'], ['사용', '하기', '편리해서', '좋아요'], ['아주', '좋아요'], ['앱', '실행', '하면', '상단', '에', '대출', '광고', '너무', '많이', '뜹니다', '표시', '눌러도', '계속', '뜹니다', '길거리', '에서', '명함', '뿌리는', '사채', '업자', '랑', '뭐', '가', '다릅니까'], ['좋아요'], ['일주일', '방문', '미션', '할', '때', '가끔', '은', '정답', '을', '맞혀도', '정답', '인정', '안되고', '그럴', '때', '도', '있는데', '즐겁게', '토스', '앱', '이용', '하고', '있습니다'], ['이번', '주', '미션', '이안', '열려요'], ['토스', '가', '작동', '하', '지

In [32]:
# 불용어 처리하는 함수
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

reviews_no_stopwords = [remove_stopwords(tokens) for tokens in okt_tokenized_reviews]
print(reviews_no_stopwords)

[['오늘', '의', '포인트', '를', '하려면', '업데이트', '를', '하라', '고', '문구', '가', '뜨는데', '업데이트', '가', '되지', '않음'], ['좋아요'], ['편리함'], ['재미있어요'], ['잘', '모르겠음'], ['수수료', '없고', '매일', '자주', '고', '다', '좋은데', '방송', '봤는데', '포인트', '안', '들어오구', 'ㅡㅡ', '업뎃', '하래', '서', '했는데', '계속', '업뎃해', '야', '이용', '가능하다', '뜨고', '결국', '업뎃', '만', '벌써', '번하는거', '같은데', '언제', '반영', '되나요'], ['매우', '좋음'], ['통장', '잔액', '을', '보려고', '할', '때', '재부팅', '느린', '건지', '잔액', '얼마', '남았는지', '확인', '하는데', '많은', '시간', '걸림'], ['재미', '삼아', '날', '마다', '앱', '열고', '하고있습니다'], ['포인트', '를', '원', '원', '주지', '말고', '원', '이라도', '주', '시오', 'ㅠㅠ'], ['좋습니다'], ['공유', '합니다'], ['사용', '하기', '편리해서', '좋아요'], ['아주', '좋아요'], ['앱', '실행', '하면', '상단', '에', '대출', '광고', '너무', '많이', '뜹니다', '표시', '눌러도', '계속', '뜹니다', '길거리', '에서', '명함', '뿌리는', '사채', '업자', '랑', '뭐', '가', '다릅니까'], ['좋아요'], ['일주일', '방문', '미션', '할', '때', '가끔', '은', '정답', '을', '맞혀도', '정답', '인정', '안되고', '그럴', '때', '도', '있는데', '즐겁게', '토스', '앱', '이용', '하고', '있습니다'], ['이번', '주', '미션', '이안', '열려요'], ['토스', '가', '작동', '하', '지

## 3.4.어간 추출 (Stemming) 및 표제어 추출 (Lemmatization)
- 한글은 영어와 달리 어미, 조사, 접사가 많고, 복잡한 문법 구조를 가진다.

In [33]:
# 형태소 추출 결과를 저장할 리스트
stemmed_reviews = []

for tokens in reviews_no_stopwords:
    stemmed_tokens = []
    for token in tokens:        # Okt 형태소 분석기의 morphs 메소드를 사용해서 각 토큰의 형태소 추출
        stemmed = okt.morphs(token, stem=True)      # 어간 추출 옵션 사용.
        stemmed_tokens.append(stemmed)

    stemmed_reviews.append(stemmed_tokens)

stemmed_reviews                          


[[['오늘'],
  ['의'],
  ['포인트'],
  ['를'],
  ['하다'],
  ['업데이트'],
  ['를'],
  ['하라'],
  ['고'],
  ['문구'],
  ['가다'],
  ['뜨다'],
  ['업데이트'],
  ['가다'],
  ['되다'],
  ['않다']],
 [['좋다']],
 [['편리하다']],
 [['재미있다']],
 [['자다'], ['모르다']],
 [['수수료'],
  ['없다'],
  ['매일'],
  ['자주'],
  ['고'],
  ['다'],
  ['좋다'],
  ['방송'],
  ['보다'],
  ['포인트'],
  ['안'],
  ['들어오다'],
  ['ㅡㅡ'],
  ['업뎃'],
  ['하래'],
  ['서다'],
  ['하다'],
  ['계속'],
  ['업뎃해'],
  ['야'],
  ['이용'],
  ['가능하다'],
  ['뜨다'],
  ['결국'],
  ['업뎃'],
  ['만'],
  ['벌써'],
  ['번하다'],
  ['같다'],
  ['언제'],
  ['반영'],
  ['되다']],
 [['매우'], ['좋다']],
 [['통장'],
  ['잔액'],
  ['을'],
  ['보다'],
  ['하다'],
  ['때'],
  ['재부팅'],
  ['느리다'],
  ['건지다'],
  ['잔액'],
  ['얼마'],
  ['남다'],
  ['확인'],
  ['하다'],
  ['많다'],
  ['시간'],
  ['걸리다']],
 [['재미'], ['삼다'], ['날'], ['마다'], ['앱'], ['열다'], ['하다']],
 [['포인트'],
  ['를'],
  ['원'],
  ['원'],
  ['주지'],
  ['말고'],
  ['원'],
  ['이라도'],
  ['주'],
  ['시오'],
  ['ㅠㅠ']],
 [['좋다']],
 [['공유'], ['하다']],
 [['사용'], ['하다'], ['편리하다'], ['좋다']],
 [['아주'], ['좋다']],
 [['앱'],
  ['실

In [34]:
def stem_tokens(tokens):
    return [okt.morphs(token, stem=True) for token in tokens]

stemmed_reviews = [stem_tokens(tokens) for tokens in reviews_no_stopwords]
stemmed_reviews

[[['오늘'],
  ['의'],
  ['포인트'],
  ['를'],
  ['하다'],
  ['업데이트'],
  ['를'],
  ['하라'],
  ['고'],
  ['문구'],
  ['가다'],
  ['뜨다'],
  ['업데이트'],
  ['가다'],
  ['되다'],
  ['않다']],
 [['좋다']],
 [['편리하다']],
 [['재미있다']],
 [['자다'], ['모르다']],
 [['수수료'],
  ['없다'],
  ['매일'],
  ['자주'],
  ['고'],
  ['다'],
  ['좋다'],
  ['방송'],
  ['보다'],
  ['포인트'],
  ['안'],
  ['들어오다'],
  ['ㅡㅡ'],
  ['업뎃'],
  ['하래'],
  ['서다'],
  ['하다'],
  ['계속'],
  ['업뎃해'],
  ['야'],
  ['이용'],
  ['가능하다'],
  ['뜨다'],
  ['결국'],
  ['업뎃'],
  ['만'],
  ['벌써'],
  ['번하다'],
  ['같다'],
  ['언제'],
  ['반영'],
  ['되다']],
 [['매우'], ['좋다']],
 [['통장'],
  ['잔액'],
  ['을'],
  ['보다'],
  ['하다'],
  ['때'],
  ['재부팅'],
  ['느리다'],
  ['건지다'],
  ['잔액'],
  ['얼마'],
  ['남다'],
  ['확인'],
  ['하다'],
  ['많다'],
  ['시간'],
  ['걸리다']],
 [['재미'], ['삼다'], ['날'], ['마다'], ['앱'], ['열다'], ['하다']],
 [['포인트'],
  ['를'],
  ['원'],
  ['원'],
  ['주지'],
  ['말고'],
  ['원'],
  ['이라도'],
  ['주'],
  ['시오'],
  ['ㅠㅠ']],
 [['좋다']],
 [['공유'], ['하다']],
 [['사용'], ['하다'], ['편리하다'], ['좋다']],
 [['아주'], ['좋다']],
 [['앱'],
  ['실

In [35]:
okt.pos(reviews[0])

[('오늘', 'Noun'),
 ('의', 'Josa'),
 ('포인트', 'Noun'),
 ('를', 'Josa'),
 ('하려면', 'Verb'),
 ('업데이트', 'Noun'),
 ('를', 'Josa'),
 ('하라', 'Noun'),
 ('고', 'Josa'),
 ('문구', 'Noun'),
 ('가', 'Josa'),
 ('뜨는데', 'Verb'),
 ('업데이트', 'Noun'),
 ('가', 'Josa'),
 ('되지', 'Verb'),
 ('않음', 'Verb')]

In [36]:
# 형용사를 저장할 리스트
adjectives = []

# 품사 태깅 및 형용사 추출
for word, pos in okt.pos(reviews[0]):
    if pos == 'Adjective':
        adjectives.append(word)

print("형용사 추출 결과 : ", adjectives)

형용사 추출 결과 :  []


In [37]:
# 명사만 추출
def extract_nouns(tokens):
    nouns_list = []
    for token in tokens:
        nouns = okt.nouns(token)
        nouns_list.extend(nouns)
    return nouns_list

In [38]:
noun_reviews = [extract_nouns(tokens) for tokens in reviews_no_stopwords]
noun_reviews

[['오늘', '의', '포인트', '를', '업데이트', '를', '하라', '고', '문구', '업데이트'],
 [],
 [],
 [],
 [],
 ['수수료',
  '매일',
  '자주',
  '고',
  '방송',
  '포인트',
  '안',
  '업뎃',
  '하래',
  '계속',
  '업뎃해',
  '이용',
  '업뎃',
  '만',
  '벌써',
  '언제',
  '반영'],
 ['매우'],
 ['통장', '잔액', '때', '재부팅', '잔액', '얼마', '확인', '시간'],
 ['재미', '날', '마다', '앱'],
 ['포인트', '를', '원', '원', '주지', '원', '주', '시오'],
 [],
 ['공유'],
 ['사용'],
 ['아주'],
 ['앱', '실행', '상단', '대출', '광고', '표시', '계속', '길거리', '명함', '사채', '업자', '뭐'],
 [],
 ['일주일',
  '방문',
  '미션',
  '때',
  '가끔',
  '은',
  '정답',
  '정답',
  '인정',
  '때',
  '도',
  '토스',
  '앱',
  '이용'],
 ['이번', '주', '미션', '이안'],
 ['토스', '작동'],
 [],
 ['업데이트', '무한', '반복', '열기', '를', '계속', '제자리', '서버', '를', '안정화', '실행'],
 ['환급',
  '준비',
  '서류',
  '안내',
  '도',
  '순서대로',
  '결제',
  '마지막',
  '서류',
  '요구',
  '네',
  '새',
  '키',
  '서류',
  '종',
  '소',
  '세',
  '경정',
  '청구',
  '장난',
  '부동산',
  '류구',
  '지도',
  '못',
  '염장',
  '돈',
  '벌',
  '엔화',
  '로',
  '장난',
  '이제',
  '환급',
  '장난',
  '질'],
 ['만',
  '보기',
  '복권',
  '만원',
  '만원',
  '기본'

## 3.5.데이터 분석(EDA)
- 빈도 분석
- 트렌드 분석
- 감성 분석
- 트렌드, 빈도, 감성 분석에 사용할 데이터 형태로 전처리 했다면 알맞은 시각화 방법을 통해 분석해봅시다.

    - 예시 : 시간대별 리뷰 추이가 궁금하다면 -> 시간대별로 데이터 프레임 만든 후, 상위 단어 추출하여 분석
    - 예시 : 전체 리뷰의 상태가 궁금하다면 -> 전체 리뷰를 합쳐서 단어 분석